In [1]:
import json
import numpy as np
import torch
from datasets import Dataset
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm import tqdm

# ============================================================
# 1. CẤU HÌNH
# ============================================================
MODEL_NAME     = "unsloth/Qwen2.5-1.5B-Instruct"  # ✅ nhẹ hơn ~3x so với Gemma 4B
MAX_SEQ_LENGTH = 1536
MAX_NEW_TOKENS = 8

TRAIN_PATH     = "dataset/train.json"
VAL_PATH       = "dataset/validation.json"
TEST_PATH      = "dataset/test.json"
OUTPUT_DIR     = "output/outputs_qwen_fact"
BEST_MODEL_DIR = "output/qwen_fact_checking_lora"
LABEL_LIST     = ["SUPPORTED", "REFUTED", "NEI"]

# ============================================================
# 2. MODEL & TOKENIZER
# ============================================================
print("🚀 Đang khởi tạo model...")
model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    load_in_4bit    = True,
    load_in_8bit    = False,
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 16,
    lora_alpha   = 32,
    lora_dropout = 0,
    bias         = "none",
    random_state = 3407,
)

# ============================================================
# 3. TOKENIZER — ✅ đổi sang qwen-2.5
# ============================================================
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# ============================================================
# 4. FORMAT DATA
# ============================================================
INSTRUCTION = (
    "Bạn là một trợ lý AI kiểm chứng thông tin chuẩn xác. "
    "Hãy đọc các đoạn văn bối cảnh và đưa ra nhãn phân loại cho câu khẳng định. "
    "Chỉ trả về đúng một từ duy nhất, không giải thích, không dấu câu: "
    "SUPPORTED, REFUTED, hoặc NEI."
)

def build_user_message(claim: str, contexts: list) -> str:
    ctx_text = "\n".join([f"Đoạn văn {i+1}: {c}" for i, c in enumerate(contexts)])
    return (
        f"{INSTRUCTION}\n\n"
        f"[Bối cảnh]:\n{ctx_text}\n\n"
        f"[Khẳng định]: {claim}\n\n"
        f"[Nhãn phân loại]:"
    )

def format_for_training(examples: dict) -> dict:
    texts = []
    for claim, contexts, label in zip(
        examples["claim"], examples["contexts"], examples["label"]
    ):
        messages = [
            {"role": "user",      "content": build_user_message(claim, contexts)},
            {"role": "assistant", "content": label},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

# ============================================================
# 5. LOAD VÀ MAP DATA
# ============================================================
print("📦 Đang nạp dữ liệu...")
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_raw = json.load(f)
with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_raw = json.load(f)
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_raw = json.load(f)

train_dataset_raw = Dataset.from_list(train_raw)
val_dataset_raw   = Dataset.from_list(val_raw)

col_names = [c for c in train_dataset_raw.column_names if c != "text"]

train_dataset = train_dataset_raw.map(format_for_training, batched=True, remove_columns=col_names)
val_dataset   = val_dataset_raw.map(format_for_training,   batched=True, remove_columns=col_names)

print(f"  ✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_raw)} mẫu")

# ============================================================
# 6. ✅ COMPUTE METRICS — in F1 val sau mỗi eval step
# ============================================================
def predict_label_batch(model, tokenizer, samples: list) -> list:
    """Inference nhanh cho validation — dùng trong compute_metrics."""
    preds = []
    FastModel.for_inference(model)
    for sample in samples:
        messages = [{"role": "user", "content": build_user_message(
            sample["claim"], sample["contexts"]
        )}]
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens = MAX_NEW_TOKENS,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip().upper()
        pred = next((l for l in LABEL_LIST if l in generated), "NEI")
        preds.append(pred)
    FastModel.for_training(model)  # trả lại chế độ train
    return preds

from transformers import TrainerCallback

class F1ValidationCallback(TrainerCallback):
    """In F1 macro trên val set sau mỗi eval step."""

    def __init__(self, val_raw, model, tokenizer, label_list):
        self.val_raw    = val_raw
        self.model      = model
        self.tokenizer  = tokenizer
        self.label_list = label_list

    def on_evaluate(self, args, state, control, **kwargs):
        print("\n📊 Đang tính F1 trên validation set...")
        y_true = [s["label"].strip().upper() for s in self.val_raw]
        y_pred = predict_label_batch(self.model, self.tokenizer, self.val_raw)

        f1_macro    = f1_score(y_true, y_pred, average="macro",    labels=self.label_list, zero_division=0)
        f1_weighted = f1_score(y_true, y_pred, average="weighted", labels=self.label_list, zero_division=0)
        accuracy    = accuracy_score(y_true, y_pred)

        print(f"  → Accuracy    : {accuracy:.4f} ({accuracy*100:.2f}%)")
        print(f"  → F1 Macro    : {f1_macro:.4f}")
        print(f"  → F1 Weighted : {f1_weighted:.4f}")
        print(f"  → Step        : {state.global_step}")

        # Ghi vào log để Trainer lưu lại
        if state.log_history is not None:
            state.log_history.append({
                "step":         state.global_step,
                "eval_f1":      f1_macro,
                "eval_f1_w":    f1_weighted,
                "eval_accuracy_cls": accuracy,
            })

# ============================================================
# 7. TRAINING
# ============================================================
print("\n🏋️  Bắt đầu training...")

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 50,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        eval_strategy               = "steps",
        eval_steps                  = 50,
        save_strategy               = "steps",
        save_steps                  = 50,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",  # vẫn dùng loss để save checkpoint
        greater_is_better           = False,
        save_total_limit            = 2,
        optim                       = "adamw_8bit",
        weight_decay                = 0.001,
        lr_scheduler_type           = "cosine",
        seed                        = 3407,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
    ),
    callbacks = [F1ValidationCallback(val_raw, model, tokenizer, LABEL_LIST)],  # ✅ thêm callback
)

# ✅ Qwen dùng <|im_start|> thay vì <start_of_turn>
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()
print(f"✅ Training xong! {trainer_stats.metrics['train_runtime']:.1f}s")

# ============================================================
# 8. LƯU MODEL
# ============================================================
model.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"💾 Đã lưu model tại: {BEST_MODEL_DIR}")

# ============================================================
# 9. INFERENCE TRÊN TEST SET
# ============================================================
print("\n🔍 Đánh giá trên tập Test...")
FastModel.for_inference(model)

def predict_label(claim: str, contexts: list) -> str:
    messages = [{"role": "user", "content": build_user_message(claim, contexts)}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip().upper()
    return next((l for l in LABEL_LIST if l in generated), "NEI")

y_true, y_pred = [], []
for sample in tqdm(test_raw, desc="Inference"):
    y_true.append(sample["label"].strip().upper())
    y_pred.append(predict_label(sample["claim"], sample["contexts"]))

# ============================================================
# 10. ĐÁNH GIÁ
# ============================================================
print("\n" + "="*60)
accuracy    = accuracy_score(y_true, y_pred)
f1_macro    = f1_score(y_true, y_pred, average="macro",    labels=LABEL_LIST, zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", labels=LABEL_LIST, zero_division=0)

print(f"Accuracy    : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1 Macro    : {f1_macro:.4f}")
print(f"F1 Weighted : {f1_weighted:.4f}")
print(classification_report(y_true, y_pred, labels=LABEL_LIST, zero_division=0))

results_path = "output/test_predictions.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump([
        {"claim": s["claim"], "true_label": t, "pred_label": p}
        for s, t, p in zip(test_raw, y_true, y_pred)
    ], f, ensure_ascii=False, indent=2)

print(f"💾 Prediction: {results_path}")
print("🎉 Hoàn tất!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🚀 Đang khởi tạo model...
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.691 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



📊 Đang tính F1 trên validation set...
  → Accuracy    : 0.4930 (49.30%)
  → F1 Macro    : 0.3318
  → F1 Weighted : 0.3950
  → Step        : 50

📊 Đang tính F1 trên validation set...
  → Accuracy    : 0.7831 (78.31%)
  → F1 Macro    : 0.7464
  → F1 Weighted : 0.7785
  → Step        : 100

📊 Đang tính F1 trên validation set...
  → Accuracy    : 0.8028 (80.28%)
  → F1 Macro    : 0.7828
  → F1 Weighted : 0.8059
  → Step        : 150

📊 Đang tính F1 trên validation set...
  → Accuracy    : 0.6535 (65.35%)
  → F1 Macro    : 0.6509
  → F1 Weighted : 0.6641
  → Step        : 200

📊 Đang tính F1 trên validation set...
  → Accuracy    : 0.8338 (83.38%)
  → F1 Macro    : 0.8158
  → F1 Weighted : 0.8329
  → Step        : 250

📊 Đang tính F1 trên validation set...
  → Accuracy    : 0.8394 (83.94%)
  → F1 Macro    : 0.8084
  → F1 Weighted : 0.8326
  → Step        : 300

📊 Đang tính F1 trên validation set...
  → Accuracy    : 0.7775 (77.75%)
  → F1 Macro    : 0.7636
  → F1 Weighted : 0.7811
  → Step

Inference: 100%|██████████| 888/888 [03:37<00:00,  4.08it/s]



Accuracy    : 0.8705 (87.05%)
F1 Macro    : 0.8566
F1 Weighted : 0.8706
              precision    recall  f1-score   support

   SUPPORTED       0.89      0.86      0.87       353
     REFUTED       0.78      0.80      0.79       175
         NEI       0.90      0.92      0.91       360

    accuracy                           0.87       888
   macro avg       0.86      0.86      0.86       888
weighted avg       0.87      0.87      0.87       888



FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/test_predictions.json'

In [ ]:
import json
import torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm import tqdm

# ============================================================
# CẤU HÌNH
# ============================================================
BEST_MODEL_DIR = "output/checkpoint-150"  # đường dẫn checkpoint
TEST_PATH      = "dataset/test.json"
LABEL_LIST     = ["SUPPORTED", "REFUTED", "NEI"]
MAX_NEW_TOKENS = 8
MAX_SEQ_LENGTH = 1536

INSTRUCTION = (
    "Bạn là một trợ lý AI kiểm chứng thông tin chuẩn xác. "
    "Hãy đọc các đoạn văn bối cảnh và đưa ra nhãn phân loại cho câu khẳng định. "
    "Chỉ trả về đúng một từ duy nhất, không giải thích, không dấu câu: "
    "SUPPORTED, REFUTED, hoặc NEI."
)

# ============================================================
# LOAD CHECKPOINT
# ============================================================
print("🚀 Đang load checkpoint...")
model, tokenizer = FastModel.from_pretrained(
    model_name     = BEST_MODEL_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
FastModel.for_inference(model)
print("✅ Load xong!")

# ============================================================
# INFERENCE
# ============================================================
def build_user_message(claim: str, contexts: list) -> str:
    ctx_text = "\n".join([f"Đoạn văn {i+1}: {c}" for i, c in enumerate(contexts)])
    return (
        f"{INSTRUCTION}\n\n"
        f"[Bối cảnh]:\n{ctx_text}\n\n"
        f"[Khẳng định]: {claim}\n\n"
        f"[Nhãn phân loại]:"
    )

def predict_label(claim: str, contexts: list) -> str:
    messages = [{"role": "user", "content": build_user_message(claim, contexts)}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.convert_tokens_to_ids("<|im_end|>"),
        )

    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip().upper()

    return next((l for l in LABEL_LIST if l in generated), "NEI")

# ============================================================
# TEST TRÊN FILE — hoặc thử 1 mẫu nhanh
# ============================================================

# --- Thử 1 mẫu nhanh để kiểm tra model hoạt động ---
print("\n🧪 Quick test:")
sample_claim    = "Hà Nội là thủ đô của Việt Nam."
sample_contexts = ["Hà Nội là thủ đô và là thành phố lớn nhất của Việt Nam."]
result = predict_label(sample_claim, sample_contexts)
print(f"  Claim   : {sample_claim}")
print(f"  Dự đoán : {result}")   # kỳ vọng: SUPPORTED

# --- Đánh giá toàn bộ test set ---
print(f"\n🔍 Đánh giá trên: {TEST_PATH}")
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_raw = json.load(f)

y_true, y_pred, raw_outputs = [], [], []
for sample in tqdm(test_raw, desc="Inference"):
    true_label = sample["label"].strip().upper()
    pred_label = predict_label(sample["claim"], sample["contexts"])
    y_true.append(true_label)
    y_pred.append(pred_label)
    raw_outputs.append({
        "claim":      sample["claim"],
        "true_label": true_label,
        "pred_label": pred_label,
        "correct":    true_label == pred_label,
    })

# ============================================================
# KẾT QUẢ
# ============================================================
print("\n" + "="*60)
accuracy    = accuracy_score(y_true, y_pred)
f1_macro    = f1_score(y_true, y_pred, average="macro",    labels=LABEL_LIST, zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", labels=LABEL_LIST, zero_division=0)

print(f"Accuracy    : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1 Macro    : {f1_macro:.4f}")
print(f"F1 Weighted : {f1_weighted:.4f}")
print()
print(classification_report(y_true, y_pred, labels=LABEL_LIST, zero_division=0))

# Lưu kết quả chi tiết
out_path = "test_predictions.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "metrics": {
            "accuracy":    round(accuracy, 4),
            "f1_macro":    round(f1_macro, 4),
            "f1_weighted": round(f1_weighted, 4),
        },
        "predictions": raw_outputs,
    }, f, ensure_ascii=False, indent=2)

print(f"💾 Đã lưu chi tiết tại: {out_path}")